# Métriques — two cubes

Erreur de rotation des objets (en degrés), loss de classification, loss totale et taux de correspondances correctes en fonction des steps.

Chaque courbe correspond à un run et un split (`train` ou `val`). Les métriques de rotation sont enregistrées en radians puis converties en degrés. Le train est lissé par moyenne glissante ; mettre `SMOOTHING = 1` pour les valeurs brutes.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Fonctionne depuis la racine du projet ou depuis notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "runs").is_dir() else Path.cwd().parent
RUNS_DIR = ROOT / "runs" / "two_cubes"
if not RUNS_DIR.exists():
    RUNS_DIR = ROOT / "runs" / "random_8_views" / "run_1"

SMOOTHING = 100  # Nombre de steps pour lisser le train.
SPLITS = ("train", "val")  # Ajouter "val_2_cubes" si souhaité.

runs = {}
for path in sorted(RUNS_DIR.rglob("metrics.jsonl")):
    with path.open() as file:
        runs[str(path.parent.relative_to(RUNS_DIR))] = [
            json.loads(line) for line in file if line.strip()
        ]
if not runs:
    raise FileNotFoundError(f"Aucun metrics.jsonl dans {RUNS_DIR}")
print(f"Dossier : {RUNS_DIR}")
for name, rows in runs.items():
    print(f"{name} : {len(rows)} mesures, splits = {sorted({r['split'] for r in rows})}")


In [ ]:
metrics = [
    ("object_rotation_angle", "Erreur de rotation des objets", "Degrés", 180 / np.pi),
    ("camera_rotation_angle", "Erreur de rotation des caméras", "Degrés", 180 / np.pi),
    ("object_class", "Loss de classification des objets", "Loss", 1),
    ("total", "Loss totale", "Loss", 1),
    ("matching_accuracy", "Correspondances correctes", "%", 100),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
for ax, (key, title, ylabel, factor) in zip(axes.flat, metrics):
    for name, rows in runs.items():
        for split in SPLITS:
            points = sorted(
                (r for r in rows if r["split"] == split and key in r),
                key=lambda r: r["step"],
            )
            if not points:
                continue
            steps = np.array([r["step"] for r in points])
            values = np.array([r[key] for r in points]) * factor
            window = min(max(1, SMOOTHING), len(values)) if split == "train" else 1
            if window > 1:
                values = np.convolve(values, np.ones(window) / window, mode="valid")
                steps = steps[window - 1:]
            ax.plot(steps, values, label=f"{name} / {split}", linewidth=1.5)
    ax.set(title=title, xlabel="Step", ylabel=ylabel)
    ax.grid(alpha=0.25)
    ax.legend()
    if key == "matching_accuracy":
        ax.set_ylim(0, 105)
plt.show()
